# Two PCB datasets, then component and defect classification

This notebook takes the two datasets the same way as the Colab notebook:

1. **HRIPCB** from Kaggle: `youssefhassan12/hripcb-dataset`
2. **WACV 2019** from `https://ripl.cc.gatech.edu/data/pcb_wacv_2019.zip`

WACV boxes come from VOC XML and are named `component_<name>`. This Kaggle copy of HRIPCB stores YOLO boxes, named `defect_<name>`. Text, pins, pads, unknown, switch, and test are dropped. The records are shuffled and written as one COCO dataset, 80% train / 10% val / 10% test.

OpenCV then finds part-like boxes on an assembled board. Each box is resized to 224 x 224. Two Mamba models are fine-tuned on the same crops and the same split. One names the component. The other names the defect. Each prediction has its own confidence.

Put `kaggle.json` in this folder, or in your user `.kaggle` folder, before the download cell. On Colab the download cell asks you to upload that file.


In [1]:
import importlib
import json
import os
import random
import re
import shutil
import stat
import subprocess
import sys
import urllib.request
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

def need(module, pip_name=None):
    try:
        importlib.import_module(module)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or module])

for module, pip_name in (("PIL", "pillow"), ("matplotlib", "matplotlib"), ("yaml", "pyyaml"), ("lxml", "lxml"), ("pandas", "pandas"), ("kaggle", "kaggle")):
    need(module, pip_name)

from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("PyTorch is not installed:", exc)


PyTorch: 2.14.1+cpu
CUDA available: False


## Download both archives

Kaggle needs `kaggle.json`. On this PC the cell finds the file beside the notebook. On Colab that file is not there, so the cell opens a file picker: choose `kaggle.json` from your computer. It is then saved to `~/.kaggle/kaggle.json`.


In [ ]:
try:
    PROJECT = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    PROJECT = Path.cwd()

BASE = PROJECT / "pcb_vim"
RAW = BASE / "raw"
EXTRACTED = BASE / "extracted"
RAW.mkdir(parents=True, exist_ok=True)
EXTRACTED.mkdir(parents=True, exist_ok=True)

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
search_dirs = [PROJECT, Path.cwd(), Path.home() / "Downloads", Path.home() / "Desktop" / "btech_proj"]
found = None
for folder in search_dirs:
    candidate = folder / "kaggle.json"
    if candidate.is_file():
        found = candidate
        break
if found is None and (kaggle_dir / "kaggle.json").is_file():
    found = kaggle_dir / "kaggle.json"
if found is None:
    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None
    if colab_files is not None:
        print("This is Colab. Choose the kaggle.json file from your computer.")
        uploaded = colab_files.upload()
        payload = uploaded.get("kaggle.json")
        if payload is None:
            for name, data in uploaded.items():
                if name.lower().endswith(".json"):
                    payload = data
                    break
        if payload:
            found = kaggle_dir / "kaggle.json"
            found.write_bytes(payload)
            os.chmod(found, 0o600)
if found is None:
    looked = ", ".join(str(folder / "kaggle.json") for folder in search_dirs)
    raise FileNotFoundError("kaggle.json was not found. Looked in: " + looked)

target = kaggle_dir / "kaggle.json"
if found.resolve() != target.resolve():
    shutil.copy2(found, target)
    os.chmod(target, stat.S_IREAD | stat.S_IWRITE)
creds = json.loads(target.read_text(encoding="utf-8"))
if not creds.get("username") or not creds.get("key"):
    raise ValueError("kaggle.json must contain username and key")
os.environ["KAGGLE_USERNAME"] = creds["username"]
os.environ["KAGGLE_KEY"] = creds["key"]
print("Using kaggle.json from", found)

def download_kaggle_zip(dest_dir, username, key):
    import base64
    dest = dest_dir / "hripcb-dataset.zip"
    if dest.is_file() and dest.stat().st_size > 1_000_000 and zipfile.is_zipfile(dest):
        print("already downloaded", dest.name)
        return
    partial = dest.with_suffix(".zip.part")
    token = base64.b64encode(f"{username}:{key}".encode()).decode()
    url = "https://www.kaggle.com/api/v1/datasets/download/youssefhassan12/hripcb-dataset"
    request = urllib.request.Request(url, headers={"Authorization": "Basic " + token, "User-Agent": "pcb-notebook"})
    try:
        response = urllib.request.urlopen(request, timeout=180)
    except Exception as exc:
        detail = ""
        if hasattr(exc, "read"):
            detail = exc.read().decode("utf-8", errors="replace")[:400]
        code = getattr(exc, "code", "")
        raise RuntimeError(f"Kaggle refused the download ({code}). {detail}".strip()) from exc
    with response, open(partial, "wb") as handle:
        total = int(response.headers.get("Content-Length") or 0)
        done = 0
        while True:
            chunk = response.read(1 << 20)
            if not chunk:
                break
            handle.write(chunk)
            done += len(chunk)
            if done % (25 << 20) < (1 << 20):
                print(f"  {done/1e6:.0f} MB", flush=True)
    if not zipfile.is_zipfile(partial):
        preview = partial.read_bytes()[:200].decode("utf-8", errors="replace")
        partial.unlink(missing_ok=True)
        raise RuntimeError("Kaggle did not return a zip. " + preview)
    partial.replace(dest)
    print("saved", dest.name, f"{dest.stat().st_size/1e6:.0f} MB")

download_kaggle_zip(RAW, creds["username"], creds["key"])

wacv_zip = RAW / "pcb_wacv_2019.zip"
wacv_url = "https://ripl.cc.gatech.edu/data/pcb_wacv_2019.zip"
if wacv_zip.exists() and wacv_zip.stat().st_size > 200_000_000:
    print("already downloaded", wacv_zip.name)
else:
    partial = wacv_zip.with_suffix(".zip.part")
    last_error = None
    for attempt in range(1, 5):
        print(f"downloading {wacv_url} (attempt {attempt}/4)")
        try:
            request = urllib.request.Request(wacv_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=180) as response, open(partial, "wb") as handle:
                total = int(response.headers.get("Content-Length") or 0)
                done = 0
                while True:
                    chunk = response.read(1 << 20)
                    if not chunk:
                        break
                    handle.write(chunk)
                    done += len(chunk)
                    if total:
                        print(f"\r  {done/1e6:.0f} / {total/1e6:.0f} MB", end="")
                print()
            if total and done != total:
                raise RuntimeError(f"stopped at {done} of {total} bytes")
            partial.replace(wacv_zip)
            last_error = None
            break
        except Exception as exc:
            last_error = exc
            print("download failed:", exc)
            partial.unlink(missing_ok=True)
    if last_error is not None:
        raise RuntimeError("could not download the WACV zip") from last_error

print("Archives downloaded:", [p.name for p in RAW.iterdir()])


## Unzip both


In [ ]:
for archive in RAW.glob("*.zip"):
    dest = EXTRACTED / archive.stem
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(dest)
    print(f"{archive.name}: extracted to {dest}")


## Read boxes from both datasets

WACV boxes come from VOC XML and are named `component_<name>`. The Kaggle HRIPCB archive stores YOLO labels (`images` plus `labels/*.txt`, class names in `data.yaml`); those boxes are named `defect_<name>`. A file is kept only when its path contains `pcb_wacv_2019` or `hripcb`.


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def clean_name(text):
    return re.sub(r"[^a-z0-9]+", "_", text.strip().lower()).strip("_")


def find_text(node, tag):
    child = node.find(tag)
    return child.text.strip() if child is not None and child.text else None


def get_box(obj):
    box = obj.find("bndbox")
    if box is None:
        return None
    try:
        x1 = float(find_text(box, "xmin"))
        y1 = float(find_text(box, "ymin"))
        x2 = float(find_text(box, "xmax"))
        y2 = float(find_text(box, "ymax"))
        return x1, y1, x2, y2
    except (TypeError, ValueError):
        return None


def parse_xml_objects(xml_path, source):
    root = ET.parse(xml_path).getroot()
    objects = root.findall("object")
    if not objects:
        for child in list(root):
            if child.find("bndbox") is not None and child.find("name") is not None:
                objects.append(child)
    parsed = []
    for obj in objects:
        name = find_text(obj, "name")
        box = get_box(obj)
        if not name or not box:
            continue
        label = clean_name(name)
        x1, y1, x2, y2 = box
        if source == "wacv":
            if label in {"text", "pins", "pads", "unknown", "switch", "test"}:
                continue
            label = "component_" + label
        else:
            label = "defect_" + label
        if x2 <= x1 or y2 <= y1:
            continue
        parsed.append((label, x1, y1, x2, y2))
    return root, parsed


image_lookup = defaultdict(list)
for path in EXTRACTED.rglob("*"):
    if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
        image_lookup[path.name.lower()].append(path)

records = []
unmatched_xml = []
unparsed_xml = []

for xml_path in EXTRACTED.rglob("*.xml"):
    source_text = str(xml_path).lower()
    if "pcb_wacv_2019" in source_text:
        source = "wacv"
    elif "hripcb" in source_text:
        source = "hripcb"
    else:
        continue
    try:
        root, objects = parse_xml_objects(xml_path, source)
    except Exception:
        unparsed_xml.append(str(xml_path))
        continue

    filename = find_text(root, "filename")
    candidates = []
    if filename:
        candidates.extend(image_lookup.get(Path(filename).name.lower(), []))
    for ext in IMAGE_EXTS:
        candidates.extend(image_lookup.get((xml_path.stem + ext).lower(), []))
    if not candidates:
        candidates = [
            path for path in xml_path.parent.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTS
        ]

    image_path = None
    for candidate in candidates:
        try:
            with Image.open(candidate) as image:
                image.verify()
            image_path = candidate
            break
        except Exception:
            pass
    if image_path is None:
        unmatched_xml.append(str(xml_path))
        continue
    try:
        with Image.open(image_path) as image:
            width, height = image.size
    except Exception:
        continue
    if objects:
        records.append({
            "source": source,
            "image": image_path,
            "width": width,
            "height": height,
            "objects": objects,
        })


import yaml


def yolo_names_for(image_path):
    for parent in [image_path.parent, *image_path.parents]:
        yaml_path = parent / "data.yaml"
        if yaml_path.is_file():
            data = yaml.safe_load(yaml_path.read_text(encoding="utf-8")) or {}
            names = data.get("names", {})
            if isinstance(names, dict):
                return {int(key): clean_name(str(value)) for key, value in names.items()}
            return {index: clean_name(str(value)) for index, value in enumerate(names)}
        if parent == EXTRACTED:
            break
    return {}


def yolo_label_file(image_path):
    beside = image_path.with_suffix(".txt")
    if beside.is_file():
        return beside
    if image_path.parent.name.lower() == "images":
        candidate = image_path.parent.parent / "labels" / f"{image_path.stem}.txt"
        if candidate.is_file():
            return candidate
    return None


def parse_yolo(label_path, width, height, names):
    parsed = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            class_id = int(float(parts[0]))
            cx, cy, bw, bh = (float(part) for part in parts[1:5])
        except ValueError:
            continue
        name = names.get(class_id)
        if not name:
            continue
        x1 = (cx - bw / 2.0) * width
        y1 = (cy - bh / 2.0) * height
        x2 = (cx + bw / 2.0) * width
        y2 = (cy + bh / 2.0) * height
        if x2 <= x1 or y2 <= y1:
            continue
        parsed.append(("defect_" + name, x1, y1, x2, y2))
    return parsed


seen_images = {record["image"].resolve() for record in records}
for image_path in EXTRACTED.rglob("*"):
    if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTS:
        continue
    if "hripcb" not in str(image_path).lower():
        continue
    if image_path.resolve() in seen_images:
        continue
    label_path = yolo_label_file(image_path)
    if label_path is None:
        continue
    try:
        with Image.open(image_path) as image:
            image.verify()
        with Image.open(image_path) as image:
            width, height = image.size
    except Exception:
        continue
    objects = parse_yolo(label_path, width, height, yolo_names_for(image_path))
    if not objects:
        continue
    records.append({
        "source": "hripcb",
        "image": image_path,
        "width": width,
        "height": height,
        "objects": objects,
    })

print("Usable annotated images:", len(records))
print("By source:", dict(Counter(record["source"] for record in records)))
print("Unmatched XML:", len(unmatched_xml))
print("Unparsed XML:", len(unparsed_xml))


## Class counts from both datasets


In [ ]:
class_counts = Counter(
    label
    for record in records
    for label, *_ in record["objects"]
)
for label, count in class_counts.most_common():
    print(f"{label:35s} {count}")


## Merge into one COCO dataset

Images are copied into `pcb_vim/combined_coco/{train,val,test}/images`. Annotations are COCO JSON. Names are prefixed with the source so the two datasets do not collide: `wacv_000000.jpg`, `hripcb_000123.jpg`.


In [ ]:
random.seed(42)
OUT = BASE / "combined_coco"
if OUT.exists():
    shutil.rmtree(OUT)
for split in ("train", "val", "test"):
    (OUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUT / split / "annotations").mkdir(parents=True, exist_ok=True)

random.shuffle(records)
n = len(records)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
split_records = {
    "train": records[:n_train],
    "val": records[n_train:n_train + n_val],
    "test": records[n_train + n_val:],
}
all_classes = sorted({obj[0] for record in records for obj in record["objects"]})
class_to_id = {name: i + 1 for i, name in enumerate(all_classes)}
print("Classes:", class_to_id)

for split, items in split_records.items():
    coco = {
        "images": [],
        "annotations": [],
        "categories": [{"id": class_to_id[name], "name": name} for name in all_classes],
    }
    annotation_id = 1
    for i, rec in enumerate(items):
        img_path = rec["image"]
        new_name = f"{rec['source']}_{i:06d}{img_path.suffix.lower()}"
        dst = OUT / split / "images" / new_name
        shutil.copy2(img_path, dst)
        image_id = i + 1
        coco["images"].append({
            "id": image_id,
            "file_name": new_name,
            "width": rec["width"],
            "height": rec["height"],
        })
        for label, x1, y1, x2, y2 in rec["objects"]:
            x1 = max(0, min(x1, rec["width"]))
            x2 = max(0, min(x2, rec["width"]))
            y1 = max(0, min(y1, rec["height"]))
            y2 = max(0, min(y2, rec["height"]))
            if x2 <= x1 or y2 <= y1:
                continue
            coco["annotations"].append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": class_to_id[label],
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "area": (x2 - x1) * (y2 - y1),
                "iscrowd": 0,
            })
            annotation_id += 1
    ann_path = OUT / split / "annotations" / "instances.json"
    with open(ann_path, "w", encoding="utf-8") as handle:
        json.dump(coco, handle)
    print(split, "images:", len(coco["images"]), "boxes:", len(coco["annotations"]), "annotations:", ann_path)

print("Unified COCO dataset:", OUT)


## One sample from each dataset


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, source in zip(axes, ("wacv", "hripcb")):
    sample = next(record for record in split_records["train"] if record["source"] == source and record["objects"])
    with Image.open(sample["image"]) as image:
        img = image.convert("RGB").copy()
    ax.imshow(img)
    for label, x1, y1, x2, y2 in sample["objects"][:40]:
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=1.2))
        ax.text(x1, max(0, y1 - 2), label, fontsize=7, color="yellow")
    ax.set_title(f"{source}  {sample['image'].name}")
    ax.axis("off")
fig.tight_layout()
plt.show()


## OpenCV finds the parts

On an assembled WACV board the box is a component. The image is turned to grayscale, blurred, and compared with a smooth background. A threshold and contours then keep boxes that look like parts: not the whole board, not a speck, and not a long thin trace.


In [ ]:
import math
need("cv2", "opencv-python")
need("torch", "torch")
need("torchvision", "torchvision")
need("sklearn", "scikit-learn")
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 40
PATIENCE = 8
IMG_SIZE = 224
SEED = 42
MAX_SIDE = 1400
if DEVICE.type == "cuda":
    MAX_PER_CLASS = 250
    BATCH = 16
    MV_CFG = dict(in_dim=32, dim=80, res_depths=(1, 3), hybrid_depths=(8, 4), hybrid_mamba=(4, 2), hybrid_heads=(8, 16), d_state=8)
else:
    MAX_PER_CLASS = 80
    BATCH = 8
    MV_CFG = dict(in_dim=24, dim=48, res_depths=(1, 2), hybrid_depths=(4, 2), hybrid_mamba=(2, 1), hybrid_heads=(4, 8), d_state=4)
print("device", DEVICE, "| epochs", EPOCHS, "| max per class", MAX_PER_CLASS)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def resize_max(image, max_side=MAX_SIDE):
    height, width = image.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    if scale >= 0.999:
        return image.copy(), 1.0
    out = cv2.resize(image, (int(width * scale), int(height * scale)), interpolation=cv2.INTER_AREA)
    return out, scale


def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area = max(0, a[2] - a[0]) * max(0, a[3] - a[1]) + max(0, b[2] - b[0]) * max(0, b[3] - b[1]) - inter
    return inter / area if area else 0.0


def nms(boxes, scores, thresh=0.35):
    order = sorted(range(len(boxes)), key=lambda i: scores[i], reverse=True)
    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) < thresh]
    return keep


def component_proposals(bgr):
    small, scale = resize_max(bgr)
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    ksize = max(3, int(max(blur.shape) / 28) | 1)
    background = cv2.medianBlur(blur, ksize)
    diff = cv2.absdiff(blur, background)
    _, thresh = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if (thresh > 0).mean() > 0.45:
        thresh = cv2.bitwise_not(thresh)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    morph = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    morph = cv2.morphologyEx(morph, cv2.MORPH_CLOSE, kernel, iterations=2)
    contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    height, width = morph.shape
    image_area = float(height * width)
    boxes, scores = [], []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h
        if area < 0.00008 * image_area or area > 0.12 * image_area or w < 10 or h < 10:
            continue
        aspect = w / float(h)
        if aspect < 0.18 or aspect > 7.5:
            continue
        if cv2.contourArea(contour) / float(area) < 0.32:
            continue
        boxes.append([x, y, x + w, y + h])
        scores.append(area)
    kept = [boxes[i] for i in nms(boxes, scores)] if boxes else []
    full = [[int(v / scale) for v in box] for box in kept]
    steps = {
        "color": cv2.cvtColor(small, cv2.COLOR_BGR2RGB),
        "gray": gray,
        "blur": blur,
        "background": background,
        "difference": diff,
        "threshold": thresh,
        "morphology": morph,
    }
    return full, steps


def clip_box(box, width, height, pad=0):
    x1 = max(0, int(box[0]) - pad)
    y1 = max(0, int(box[1]) - pad)
    x2 = min(width, int(box[2]) + pad)
    y2 = min(height, int(box[3]) + pad)
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2, y2]


def crop_box(bgr, box, pad=2):
    height, width = bgr.shape[:2]
    clipped = clip_box(box, width, height, pad)
    if clipped is None:
        return None
    x1, y1, x2, y2 = clipped
    crop = bgr[y1:y2, x1:x2]
    return crop if crop.size else None


def to_224(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return rgb, gray


def show_row(images, titles, width=16):
    count = len(images)
    if count == 0:
        print("nothing to show")
        return
    fig, axes = plt.subplots(1, count, figsize=(width, 3.3))
    if count == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        if getattr(image, "ndim", 0) == 2:
            ax.imshow(image, cmap="gray", vmin=0, vmax=255)
        else:
            ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    plt.show()


wacv_records = [record for record in records if record["source"] == "wacv"]
if not wacv_records:
    raise RuntimeError("No WACV images were read, so OpenCV has no assembled board.")
board = cv2.imread(str(wacv_records[0]["image"]))
if board is None:
    raise RuntimeError("Could not read " + str(wacv_records[0]["image"]))
proposals, steps = component_proposals(board)
print(f"OpenCV part boxes: {len(proposals)} on {wacv_records[0]['image'].name}")
show_row([steps["color"], steps["gray"], steps["blur"]], ["Color", "Grayscale", "Blurred gray"])
show_row([steps["background"], steps["difference"], steps["threshold"]], ["Background", "Difference", "Threshold"])
show_row([steps["morphology"]], ["Morphology, then part boxes are kept"])
preview, scale = resize_max(board)
preview = cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)
for box in proposals:
    x1, y1, x2, y2 = [int(v * scale) for v in box]
    cv2.rectangle(preview, (x1, y1), (x2, y2), (230, 90, 20), 2)
show_row([preview], [f"Kept part boxes ({len(proposals)})"], width=8)


## Crop each box to 224 x 224

Color and grayscale tiles are the classifier input size. Training uses the labeled boxes from both datasets so the class name is known. OpenCV boxes are what the models are shown at the end.


In [ ]:
tiles_c, tiles_g, tile_names = [], [], []
for label, x1, y1, x2, y2 in wacv_records[0]["objects"]:
    crop = crop_box(board, [x1, y1, x2, y2], pad=3)
    if crop is None:
        continue
    rgb, gray = to_224(crop)
    tiles_c.append(rgb)
    tiles_g.append(gray)
    tile_names.append(label.replace("component_", ""))
    if len(tiles_c) == 8:
        break
print("Each tile is", IMG_SIZE, "x", IMG_SIZE)
show_row(tiles_c, ["color 224 | " + name for name in tile_names])
show_row(tiles_g, ["gray 224 | " + name for name in tile_names])


## Same crops, same split, both datasets

Two label sets are stored on every crop:

- **Component:** the WACV part name, or `none` when the crop is a bare-board HRIPCB defect or a removed part.
- **Defect:** `ok`, `missing`, `rotated`, `misplaced`, `flipped`, or the real HRIPCB defect (`defect_missing_hole`, `defect_short`, and the rest).

Rotated, missing, misplaced, and flipped are made from real WACV component crops, because those four conditions are not labels in the public files. HRIPCB keeps its real defect boxes. Train and validation follow the COCO split already built.


In [ ]:
def blank_center(bgr):
    rgb, _ = to_224(bgr)
    out = rgb.copy()
    margin = 28
    border = np.concatenate([
        out[:margin].reshape(-1, 3),
        out[-margin:].reshape(-1, 3),
        out[:, :margin].reshape(-1, 3),
        out[:, -margin:].reshape(-1, 3),
    ])
    out[margin:-margin, margin:-margin] = np.median(border, axis=0).astype(np.uint8)
    return out


def orient_rgb(rgb, kind):
    if kind == "rotated":
        turned = cv2.rotate(rgb, cv2.ROTATE_90_CLOCKWISE)
        return cv2.resize(turned, (IMG_SIZE, IMG_SIZE))
    if kind == "flipped":
        return cv2.flip(rgb, 1)
    return rgb


def misplaced_rgb(part_rgb, background_rgb):
    canvas = cv2.resize(background_rgb, (IMG_SIZE, IMG_SIZE))
    part = cv2.resize(part_rgb, (110, 70))
    canvas[70:140, 50:160] = part
    return canvas


rng = random.Random(SEED)
train_records = list(split_records["train"])
val_records = list(split_records["val"]) + list(split_records["test"])


def build_samples(source_records, limit, rng):
    samples = []
    counts = Counter()
    parts = []

    def add(rgb, component, defect):
        if counts[(component, defect)] >= limit:
            return
        counts[(component, defect)] += 1
        samples.append({"rgb": rgb, "component": component, "defect": defect})

    for record in source_records:
        image = cv2.imread(str(record["image"]))
        if image is None:
            continue
        for label, x1, y1, x2, y2 in record["objects"]:
            crop = crop_box(image, [x1, y1, x2, y2], pad=3)
            if crop is None or min(crop.shape[:2]) < 8:
                continue
            rgb, _ = to_224(crop)
            if record["source"] == "wacv":
                add(rgb, label, "ok")
                add(orient_rgb(rgb, "rotated"), label, "rotated")
                add(orient_rgb(rgb, "flipped"), label, "flipped")
                add(blank_center(crop), "none", "missing")
                parts.append((rgb, label))
            else:
                add(rgb, "none", label)
    rng.shuffle(parts)
    for index, (rgb, label) in enumerate(parts):
        if counts[(label, "misplaced")] >= limit:
            continue
        background = parts[(index + len(parts) // 2) % len(parts)][0]
        add(misplaced_rgb(rgb, background), label, "misplaced")
    rng.shuffle(samples)
    return samples


train_samples = build_samples(train_records, MAX_PER_CLASS, rng)
val_samples = build_samples(val_records, max(20, MAX_PER_CLASS // 4), random.Random(SEED + 1))
COMP_LABELS = sorted({sample["component"] for sample in train_samples})
DEFECT_LABELS = sorted({sample["defect"] for sample in train_samples})
COMP_TO_IDX = {name: i for i, name in enumerate(COMP_LABELS)}
DEFECT_TO_IDX = {name: i for i, name in enumerate(DEFECT_LABELS)}
train_samples = [sample for sample in train_samples if sample["component"] in COMP_TO_IDX and sample["defect"] in DEFECT_TO_IDX]
val_samples = [sample for sample in val_samples if sample["component"] in COMP_TO_IDX and sample["defect"] in DEFECT_TO_IDX]
if not train_samples or not val_samples:
    raise RuntimeError("No training crops were built. Run the download and merge cells first.")
print("train crops", len(train_samples), "| val crops", len(val_samples))
print("component classes", len(COMP_LABELS), COMP_LABELS)
print("defect classes", len(DEFECT_LABELS), DEFECT_LABELS)


## Two Mamba models

Both are the same hybrid: a convolutional stem, residual stages, then Mamba mixers followed by transformer blocks. They see the same 224 x 224 crops and the same train/validation split. One names the component. The other names the defect. On a GPU the depths match MambaVision-T. On CPU the same order is kept at a smaller width so 40 epochs can finish. The saved weights are the epoch with the best validation accuracy.


In [ ]:
def selective_scan(u, dt, A, B, C, D):
    batch, length, dim = u.shape
    hidden = u.new_zeros(batch, dim, A.shape[1])
    outputs = []
    for t in range(length):
        dt_t = dt[:, t].unsqueeze(-1)
        hidden = torch.exp(dt_t * A) * hidden + (dt_t * B[:, t].unsqueeze(1)) * u[:, t].unsqueeze(-1)
        outputs.append((hidden * C[:, t].unsqueeze(1)).sum(-1) + D * u[:, t])
    return torch.stack(outputs, dim=1)


class MambaVisionMixer(nn.Module):
    def __init__(self, dim, d_state=8):
        super().__init__()
        self.half = dim // 2
        self.d_state = d_state
        self.dt_rank = max(1, math.ceil(self.half / 16))
        self.in_proj = nn.Linear(dim, dim, bias=False)
        self.conv_x = nn.Conv1d(self.half, self.half, 3, padding=1, groups=self.half)
        self.conv_z = nn.Conv1d(self.half, self.half, 3, padding=1, groups=self.half)
        self.x_proj = nn.Linear(self.half, self.dt_rank + d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.half)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.half, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.half))
        self.out_proj = nn.Linear(dim, dim, bias=False)
        nn.init.zeros_(self.out_proj.weight)

    def forward(self, x):
        x_branch, z = self.in_proj(x).split(self.half, dim=-1)
        x_branch = F.silu(self.conv_x(x_branch.transpose(1, 2)).transpose(1, 2))
        z = F.silu(self.conv_z(z.transpose(1, 2)).transpose(1, 2))
        dt, B, C = torch.split(self.x_proj(x_branch), [self.dt_rank, self.d_state, self.d_state], dim=-1)
        y = selective_scan(x_branch, F.softplus(self.dt_proj(dt)), -torch.exp(self.A_log.float()), B, C, self.D.float())
        return self.out_proj(torch.cat([y, z], dim=-1))


class FeedForward(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)


class MambaBlock(nn.Module):
    def __init__(self, dim, d_state):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.mixer = MambaVisionMixer(dim, d_state)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = FeedForward(dim)

    def forward(self, x):
        b, c, h, w = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        tokens = tokens + self.mixer(self.norm1(tokens))
        tokens = tokens + self.mlp(self.norm2(tokens))
        return tokens.transpose(1, 2).reshape(b, c, h, w)


class AttentionBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = FeedForward(dim)
        nn.init.zeros_(self.attn.out_proj.weight)
        nn.init.zeros_(self.attn.out_proj.bias)

    def forward(self, x):
        b, c, h, w = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normed = self.norm1(tokens)
        attn, _ = self.attn(normed, normed, normed, need_weights=False)
        tokens = tokens + attn
        tokens = tokens + self.mlp(self.norm2(tokens))
        return tokens.transpose(1, 2).reshape(b, c, h, w)


class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv1 = nn.Conv2d(dim, dim, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(dim)
        self.conv2 = nn.Conv2d(dim, dim, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(dim)

    def forward(self, x):
        h = F.relu(self.bn1(self.conv1(x)), inplace=False)
        return F.relu(x + self.bn2(self.conv2(h)), inplace=False)


class MambaClassifier(nn.Module):
    def __init__(self, num_classes, in_dim, dim, res_depths, hybrid_depths, hybrid_mamba, hybrid_heads, d_state):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, in_dim, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(in_dim), nn.ReLU(inplace=True),
            nn.Conv2d(in_dim, dim, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(dim), nn.ReLU(inplace=True),
        )
        channels = [dim, dim * 2, dim * 4, dim * 8]
        self.stage1 = nn.Sequential(*[ResBlock(channels[0]) for _ in range(res_depths[0])])
        self.down2 = nn.Sequential(nn.Conv2d(channels[0], channels[1], 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(channels[1]), nn.ReLU(inplace=True))
        self.stage2 = nn.Sequential(*[ResBlock(channels[1]) for _ in range(res_depths[1])])
        self.down3 = nn.Sequential(nn.Conv2d(channels[1], channels[2], 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(channels[2]), nn.ReLU(inplace=True))
        self.stage3 = self._hybrid(channels[2], hybrid_depths[0], hybrid_mamba[0], hybrid_heads[0], d_state)
        self.down4 = nn.Sequential(nn.Conv2d(channels[2], channels[3], 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(channels[3]), nn.ReLU(inplace=True))
        self.stage4 = self._hybrid(channels[3], hybrid_depths[1], hybrid_mamba[1], hybrid_heads[1], d_state)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(channels[3], num_classes)

    @staticmethod
    def _hybrid(dim, depth, n_mamba, heads, d_state):
        blocks = []
        for i in range(depth):
            blocks.append(MambaBlock(dim, d_state) if i < n_mamba else AttentionBlock(dim, heads))
        return nn.Sequential(*blocks)

    def forward(self, x):
        x = self.stage1(self.stem(x))
        x = self.stage2(self.down2(x))
        x = self.stage3(self.down3(x))
        x = self.stage4(self.down4(x))
        return self.fc(self.drop(self.pool(x).flatten(1)))


class CropSet(Dataset):
    def __init__(self, samples, label_key, label_to_idx):
        self.samples = samples
        self.label_key = label_key
        self.label_to_idx = label_to_idx
        self.tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        return self.tf(sample["rgb"]), self.label_to_idx[sample[self.label_key]]


def run_epoch(model, loader, optimizer, criterion, train):
    model.train(train)
    total, correct, seen = 0.0, 0, 0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        if train and images.size(0) < 2:
            continue
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, targets)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total += loss.item() * images.size(0)
        correct += (logits.argmax(1) == targets).sum().item()
        seen += images.size(0)
    return total / max(1, seen), correct / max(1, seen)


def train_mamba(samples_train, samples_val, label_key, labels, label_to_idx, title, ckpt):
    ckpt = Path(ckpt)
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    if ckpt.exists():
        blob = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        model = MambaClassifier(num_classes=len(blob["labels"]), **MV_CFG).to(DEVICE)
        model.load_state_dict(blob["state"])
        model.eval()
        print(f"loaded {ckpt.name} | best val acc {blob['best_acc']:.3f} at epoch {blob['best_epoch']}")
        return model, blob["history"], blob["labels"]
    train_loader = DataLoader(CropSet(samples_train, label_key, label_to_idx), batch_size=BATCH, shuffle=True, num_workers=0)
    val_loader = DataLoader(CropSet(samples_val, label_key, label_to_idx), batch_size=BATCH, shuffle=False, num_workers=0)
    model = MambaClassifier(num_classes=len(labels), **MV_CFG).to(DEVICE)
    print(f"{title}: {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters, up to {EPOCHS} epochs")
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    history = {"train_acc": [], "val_acc": [], "val_loss": []}
    best_acc, best_epoch, stale, best_state = -1.0, 0, 0, None
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, criterion, True)
        va_loss, va_acc = run_epoch(model, val_loader, optimizer, criterion, False)
        schedule.step()
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)
        history["val_loss"].append(va_loss)
        print(f"{title} epoch {epoch:02d}/{EPOCHS}  train acc {tr_acc:.3f}  val acc {va_acc:.3f}  val loss {va_loss:.3f}")
        if va_acc > best_acc:
            best_acc, best_epoch, stale = va_acc, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"early stop at epoch {epoch}; best epoch {best_epoch} acc {best_acc:.3f}")
                break
    if best_state is None:
        raise RuntimeError(title + " did not complete an epoch")
    model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    torch.save({"state": best_state, "labels": labels, "history": history, "best_acc": best_acc, "best_epoch": best_epoch, "cfg": MV_CFG}, ckpt)
    return model, history, labels


def plot_history(history, title):
    epochs = range(1, len(history["val_acc"]) + 1)
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.plot(list(epochs), history["train_acc"], label="train")
    ax.plot(list(epochs), history["val_acc"], label="val")
    ax.set_ylim(0, 1)
    ax.set_title(title + " accuracy")
    ax.legend()
    fig.tight_layout()
    plt.show()


@torch.no_grad()
def predict_all(model, samples, label_key, label_to_idx):
    loader = DataLoader(CropSet(samples, label_key, label_to_idx), batch_size=BATCH, shuffle=False, num_workers=0)
    model.eval()
    probs, preds, trues = [], [], []
    for images, targets in loader:
        prob = torch.softmax(model(images.to(DEVICE)), dim=1).cpu().numpy()
        probs.append(prob)
        preds.append(prob.argmax(1))
        trues.append(targets.numpy())
    return np.concatenate(trues), np.concatenate(preds), np.concatenate(probs)


def confidence_report(y_true, y_pred, probs, labels, title):
    confidence = probs.max(1)
    correct = y_true == y_pred
    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"{title} accuracy {acc:.3f} | macro F1 {macro:.3f}")
    if correct.any():
        print(f"mean confidence when correct {confidence[correct].mean():.3f}")
    if (~correct).any():
        print(f"mean confidence when wrong {confidence[~correct].mean():.3f}")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    axes[0].imshow(cm, cmap="Blues")
    axes[0].set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    axes[0].set_yticks(range(len(labels)), labels)
    axes[0].set_title(title + " confusion")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    if (~correct).any():
        axes[1].hist(confidence[correct], bins=15, alpha=0.75, label="correct")
        axes[1].hist(confidence[~correct], bins=15, alpha=0.75, label="wrong")
    else:
        axes[1].hist(confidence, bins=15, label="confidence")
    axes[1].set_title(title + " confidence")
    axes[1].legend()
    fig.tight_layout()
    plt.show()
    return {"accuracy": acc, "macro_f1": macro, "mean_confidence": float(confidence.mean())}


## Train both models

The component model is trained first, then the defect model. Each runs up to 40 epochs and reloads the epoch with the best validation accuracy. If a saved weight file is already present, that file is loaded instead of training again.


In [ ]:
OUT_DIR = PROJECT / "outputs"
COMP_MODEL, comp_history, COMP_LABELS = train_mamba(
    train_samples, val_samples, "component", COMP_LABELS, COMP_TO_IDX,
    "Component Mamba", OUT_DIR / "mamba_component.pt",
)
COMP_TO_IDX = {name: i for i, name in enumerate(COMP_LABELS)}
plot_history(comp_history, "Component Mamba")
DEFECT_MODEL, defect_history, DEFECT_LABELS = train_mamba(
    train_samples, val_samples, "defect", DEFECT_LABELS, DEFECT_TO_IDX,
    "Defect Mamba", OUT_DIR / "mamba_defect.pt",
)
DEFECT_TO_IDX = {name: i for i, name in enumerate(DEFECT_LABELS)}
plot_history(defect_history, "Defect Mamba")


## Confidence for both tasks

Component confidence is the top softmax score of the component Mamba. Defect confidence is the top softmax score of the defect Mamba. They are separate numbers.


In [ ]:
comp_true, comp_pred, comp_probs = predict_all(COMP_MODEL, val_samples, "component", COMP_TO_IDX)
defect_true, defect_pred, defect_probs = predict_all(DEFECT_MODEL, val_samples, "defect", DEFECT_TO_IDX)
COMP_METRICS = confidence_report(comp_true, comp_pred, comp_probs, COMP_LABELS, "Component")
DEFECT_METRICS = confidence_report(defect_true, defect_pred, defect_probs, DEFECT_LABELS, "Defect")
print(
    f"Component accuracy {COMP_METRICS['accuracy']:.3f}, mean confidence {COMP_METRICS['mean_confidence']:.3f}. "
    f"Defect accuracy {DEFECT_METRICS['accuracy']:.3f}, mean confidence {DEFECT_METRICS['mean_confidence']:.3f}."
)


## OpenCV boxes, then both predictions

Each kept part box is resized to 224 x 224 and passed through both Mamba models. The table is the component, its confidence, the defect, and that confidence. A few real HRIPCB boxes are scored the same way.


In [ ]:
try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


@torch.no_grad()
def classify_rgb(model, labels, rgb):
    tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])(rgb).unsqueeze(0).to(DEVICE)
    prob = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    index = int(prob.argmax())
    return labels[index], float(prob[index])


rows = []
color_tiles, gray_tiles = [], []
for box in proposals[:12]:
    crop = crop_box(board, box, pad=3)
    if crop is None:
        continue
    rgb, gray = to_224(crop)
    comp_name, comp_conf = classify_rgb(COMP_MODEL, COMP_LABELS, rgb)
    defect_name, defect_conf = classify_rgb(DEFECT_MODEL, DEFECT_LABELS, rgb)
    rows.append({
        "component": comp_name,
        "component_confidence": round(comp_conf, 3),
        "defect": defect_name,
        "defect_confidence": round(defect_conf, 3),
    })
    color_tiles.append(rgb)
    gray_tiles.append(gray)

show_row(color_tiles, [f"{row['component']} {row['component_confidence']:.2f}" for row in rows])
show_row(gray_tiles, [f"{row['defect']} {row['defect_confidence']:.2f}" for row in rows])
display(pd.DataFrame(rows))

hri_records = [record for record in records if record["source"] == "hripcb"]
if hri_records:
    hri = hri_records[0]
    hri_bgr = cv2.imread(str(hri["image"]))
    hri_rows = []
    for label, x1, y1, x2, y2 in hri["objects"][:6]:
        crop = crop_box(hri_bgr, [x1, y1, x2, y2], pad=3)
        if crop is None:
            continue
        rgb, _ = to_224(crop)
        comp_name, comp_conf = classify_rgb(COMP_MODEL, COMP_LABELS, rgb)
        defect_name, defect_conf = classify_rgb(DEFECT_MODEL, DEFECT_LABELS, rgb)
        hri_rows.append({
            "xml_or_yolo_defect": label,
            "component": comp_name,
            "component_confidence": round(comp_conf, 3),
            "defect": defect_name,
            "defect_confidence": round(defect_conf, 3),
        })
    print("HRIPCB image", hri["image"].name)
    display(pd.DataFrame(hri_rows))
